In [ ]:
!pip install dash --ignore-installed blinker

In [2]:
import dash
from dash import dcc, html
import plotly.graph_objects as go
# ===================== SETUP =====================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown
import json
from IPython.display import display, clear_output
import requests

# Load data
dataset = pd.read_csv("https://huggingface.co/datasets/IshraNaznin/nyc311data2022to2024/resolve/main/data_2022_2024.csv")
geojson_path = "https://huggingface.co/datasets/IshraNaznin/nyc311data2022to2024/resolve/main/nyc-zip-code-tabulation-areas-polygons.geojson"

geojson = requests.get(geojson_path).json()

# Basic preprocessing
dataset["Created Date"] = pd.to_datetime(dataset["Created Date"], errors="coerce")
dataset["Closed Date"] = pd.to_datetime(dataset["Closed Date"], errors="coerce")

dataset["Hour"] = dataset["Created Date"].dt.hour
dataset["Day"] = dataset["Created Date"].dt.day_name()
dataset["Month"] = dataset["Created Date"].dt.month

# Drop missing key values
dataset = dataset.dropna(subset=["Complaint Type", "Borough", "Latitude", "Longitude"])

app = dash.Dash(__name__)

# ── modify each task to return fig instead of fig.show() ──
def task1():
    boroughs = ["All"] + sorted(dataset["Borough"].dropna().unique().tolist())

    fig = go.Figure()

    for borough in boroughs:
        if borough == "All":
            filtered = dataset
        else:
            filtered = dataset[dataset["Borough"] == borough]

        top = filtered["Complaint Type"].value_counts().head(10).reset_index()
        top.columns = ["Complaint Type", "Count"]

        fig.add_trace(go.Bar(
            x=top["Complaint Type"],
            y=top["Count"],
            name=borough,
            visible=(borough == "All"),
            marker=dict(
            color=top["Count"],
            colorscale="YlOrBr"  # yellow → orange → brown
        )
        ))

    buttons = []
    for i, borough in enumerate(boroughs):
        visibility = [False] * len(boroughs)
        visibility[i] = True
        buttons.append(dict(
            label=borough,
            method="update",
            args=[{"visible": visibility},
                  {"title": f" Complaint Types - {borough}"}]
        ))

    fig.update_layout(
        title="Complaint Types - All",
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.6,
            xanchor="right",
            y=1.15,
            yanchor="top",
            pad={"r": 20, "t": 10}
        )],
        xaxis_title="Complaint Type",
        yaxis_title="Count"
    )

    return fig  # ← change this line

# ... repeat for task2 through task9 ...

def task2():
    dataset_temp = dataset.copy()
    dataset_temp["Year"] = dataset_temp["Created Date"].dt.year
    dataset_temp["Month"] = dataset_temp["Created Date"].dt.month

    years = sorted(dataset_temp["Year"].dropna().unique().tolist())
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig = go.Figure()

    for year in years:
        yearly = dataset_temp[dataset_temp["Year"] == year]
        monthly = yearly.groupby("Month").size().reindex(range(1, 13), fill_value=0)

        fig.add_trace(go.Scatter(
            x=month_names,
            y=monthly.values,
            mode="lines+markers",
            name=str(year),
            visible=True
        ))

    # Build toggle buttons — one per year
    buttons = []

    # "All Years" button
    buttons.append(dict(
        label="All Years",
        method="update",
        args=[{"visible": [True] * len(years)},
              {"title": "Complaints Over Time - All Years"}]
    ))

    # Individual year buttons
    for i, year in enumerate(years):
        visibility = [False] * len(years)
        visibility[i] = True
        buttons.append(dict(
            label=str(year),
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Complaints Over Time - {year}"}]
        ))

    fig.update_layout(
        title="Complaints Over Time - All Years",
        xaxis_title="Month",
        yaxis_title="Number of Complaints",
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )],
        legend_title="Year"
    )

    return fig



def task3():
    df_temp = dataset.copy()

    # Calculate resolution time in days
    df_temp["Created Date"]    = pd.to_datetime(df_temp["Created Date"], errors="coerce")
    df_temp["Closed Date"]     = pd.to_datetime(df_temp["Closed Date"],  errors="coerce")
    df_temp["Resolution Days"] = (df_temp["Closed Date"] - df_temp["Created Date"]).dt.days
    df_temp["Year"]            = df_temp["Created Date"].dt.year

    # Drop negatives and nulls
    df_temp = df_temp[df_temp["Resolution Days"] >= 0].dropna(subset=["Resolution Days", "Agency", "Year"])

    # Filter only required years
    years = [2022, 2023, 2024]
    df_temp = df_temp[df_temp["Year"].isin(years)]

    fig = go.Figure()

    # ── One trace per year ───────────────────────────────────────
    for i, year in enumerate(years):
        df_year = df_temp[df_temp["Year"] == year]

        agg = df_year.groupby("Agency")["Resolution Days"].agg(
            Avg_Response_Time="mean",
            Total_Complaints="count",
            Slowest_Resolution="max"
        ).reset_index()
        agg["Avg_Response_Time"]  = agg["Avg_Response_Time"].round(2)
        agg["Slowest_Resolution"] = agg["Slowest_Resolution"].round(2)
        agg = agg.sort_values("Avg_Response_Time")

        fig.add_trace(go.Bar(
            x=agg["Agency"],
            y=agg["Avg_Response_Time"],
            name=str(year),
            visible=(i == 0),
            marker=dict(
                color=agg["Avg_Response_Time"],
                colorscale="YlOrRd",
                showscale=True,
                colorbar=dict(title="Avg Days")
            ),
            customdata=agg[["Avg_Response_Time", "Total_Complaints", "Slowest_Resolution"]].values,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Avg Response Time: %{customdata[0]:.2f} days<br>"
                "Total Complaints: %{customdata[1]:,}<br>"
                "Slowest Resolution: %{customdata[2]:.2f} days"
                "<extra></extra>"
            )
        ))

    # ── Dropdown buttons — one per year ──────────────────────────
    buttons = []
    for i, year in enumerate(years):
        visibility = [False] * len(years)
        visibility[i] = True
        buttons.append(dict(
            label=str(year),
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Agency Efficiency — Average Response Time ({year})"}
            ]
        ))

    fig.update_layout(
        title=f"Agency Efficiency — Average Response Time ({years[0]})",
        xaxis_title="Agency",
        yaxis_title="Average Days to Resolve",
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )],
        xaxis=dict(tickangle=-45),
        height=600,
        margin=dict(b=150)
    )

    return fig


def task4():
    boroughs = sorted(dataset["Borough"].dropna().unique().tolist())

    fig = go.Figure()

    # Add one Treemap trace per Borough
    for i, borough in enumerate(boroughs):
        temp = dataset[dataset["Borough"] == borough]
        temp = temp.groupby("Complaint Type").size().reset_index(name="Count")
        temp = temp.sort_values("Count", ascending=False)

        fig.add_trace(go.Treemap(
            labels=temp["Complaint Type"].tolist(),
            parents=[""] * len(temp),
            values=temp["Count"].tolist(),
            visible=(i == 0),
            name=borough,
            textinfo="label+value+percent root"
        ))

    # Build dropdown buttons
    buttons = []
    for i, borough in enumerate(boroughs):
        visibility = [False] * len(boroughs)
        visibility[i] = True
        buttons.append(dict(
            label=borough,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Complaint Type Distribution in {borough}"}
            ]
        ))

    fig.update_layout(
        title=f"Complaint Type Distribution in {boroughs[0]}",
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )]
    )

    return fig


def task5():
    temp = dataset.groupby(["Borough", "Complaint Type"]).size().reset_index(name="Count")
    top_types = dataset["Complaint Type"].value_counts().head(5).index
    temp = temp[temp["Complaint Type"].isin(top_types)]

    fig = px.bar(temp, x="Borough", y="Count", color="Complaint Type",
                 title="Stacked Complaints by Borough")
    return fig



def task6():
    top_complaints = dataset["Complaint Type"].value_counts().head(20).index.tolist()

    default_complaint = top_complaints[0]
    temp = dataset[dataset["Complaint Type"] == default_complaint]
    zip_counts = temp.groupby("Incident Zip").size().reset_index(name="Count")

    # Merge zip with borough/city info for hover
    zip_info = dataset[["Incident Zip", "Borough"]].drop_duplicates()
    zip_counts = zip_counts.merge(zip_info, on="Incident Zip", how="left")

    fig = go.Figure(go.Choroplethmapbox(
        geojson=geojson,
        locations=zip_counts["Incident Zip"],
        featureidkey="properties.postalCode",
        z=zip_counts["Count"],
        colorscale="YlOrRd",          # light yellow → orange → red for density
        zmin=0,
        zmax=400,
        colorbar=dict(title="Complaints"),
        showscale=True,
        marker_opacity=0.8,
        marker_line_width=0.5,
        customdata=zip_counts[["Borough", "Incident Zip"]].values,
        hovertemplate=(
            "<b>%{customdata[1]}</b><br>"
            "Borough: %{customdata[0]}<br>"
            "Complaints: %{z}<extra></extra>"
        ),
        name=default_complaint
    ))

    # Build dropdown buttons
    buttons = []
    for complaint in top_complaints:
        temp = dataset[dataset["Complaint Type"] == complaint]
        zip_counts = temp.groupby("Incident Zip").size().reset_index(name="Count")
        zip_counts = zip_counts.merge(zip_info, on="Incident Zip", how="left")

        buttons.append(dict(
            label=complaint,
            method="update",
            args=[
                {
                    "locations":    [zip_counts["Incident Zip"].tolist()],
                    "z":            [zip_counts["Count"].tolist()],
                    "zmin":         [0],
                    "zmax":         [400],
                    "customdata":   [zip_counts[["Borough", "Incident Zip"]].values.tolist()],
                },
                {"title": f"Choropleth Map - {complaint}"}
            ]
        ))

    fig.update_layout(
        title=f"Choropleth Map - {default_complaint}",
        mapbox=dict(
        style="carto-positron",
        center={"lat": 40.7, "lon": -73.9},
        zoom=9
        ),
        margin={"r":0,"t":50,"l":0,"b":0},
        mapbox_style="carto-positron",
        mapbox_center={"lat": 40.7, "lon": -73.9},
        mapbox_zoom=9,
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )]
    )

    return fig


def task7():
    df_temp = dataset.copy()
    df_temp["Created Date"] = pd.to_datetime(df_temp["Created Date"], errors="coerce")
    df_temp["Year"]         = df_temp["Created Date"].dt.year.astype("Int64")

    # Keep relevant columns, drop nulls
    df_temp = df_temp.dropna(subset=["Borough", "Status", "Year", "Complaint Type"])
    df_temp["Borough"]        = df_temp["Borough"].str.title()
    df_temp["Status"]         = df_temp["Status"].str.title()
    df_temp["Complaint Type"] = df_temp["Complaint Type"].str.title()

    years          = ["All"] + sorted(df_temp["Year"].dropna().unique().tolist())
    complaint_types = ["All"] + sorted(df_temp["Complaint Type"].dropna().unique().tolist())
    statuses       = sorted(df_temp["Status"].unique().tolist())
    boroughs       = sorted(df_temp["Borough"].unique().tolist())

    status_colors = {
        "Closed":  "#2ecc71",
        "Open":    "#e74c3c",
        "Pending": "#f39c12",
        "In Progress": "#3498db",
    }

    def build_traces(year, complaint):
        filtered = df_temp.copy()
        if year != "All":
            filtered = filtered[filtered["Year"] == int(year)]
        if complaint != "All":
            filtered = filtered[filtered["Complaint Type"] == complaint]

        agg = filtered.groupby(["Borough", "Status"]).size().reset_index(name="Count")
        total_per_borough = agg.groupby("Borough")["Count"].transform("sum")
        agg["Percent"] = (agg["Count"] / total_per_borough * 100).round(2)

        traces = []
        for status in statuses:
            df_s = agg[agg["Status"] == status]
            # Align to all boroughs
            borough_data = {b: {"Count": 0, "Percent": 0.0} for b in boroughs}
            for _, row in df_s.iterrows():
                borough_data[row["Borough"]] = {
                    "Count":   row["Count"],
                    "Percent": row["Percent"]
                }

            counts   = [borough_data[b]["Count"]   for b in boroughs]
            percents = [borough_data[b]["Percent"] for b in boroughs]

            traces.append(go.Bar(
                x=boroughs,
                y=counts,
                name=status,
                marker_color=status_colors.get(status, "#95a5a6"),
                customdata=list(zip(percents, counts)),
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    f"Status: {status}<br>"
                    "Count: %{customdata[1]:,}<br>"
                    "Share: %{customdata[0]:.2f}%"
                    "<extra></extra>"
                )
            ))
        return traces

    # ── Initial traces (All / All) ───────────────────────────────
    init_traces = build_traces("All", "All")
    fig = go.Figure(data=init_traces)

    # ── Year dropdown ────────────────────────────────────────────
    year_buttons = []
    for year in years:
        traces = build_traces(year, "All")
        year_buttons.append(dict(
            label=str(year),
            method="restyle",
            args=[{
                "x":          [t.x          for t in traces],
                "y":          [t.y          for t in traces],
                "customdata": [t.customdata for t in traces],
            }]
        ))

    # ── Complaint Type dropdown ──────────────────────────────────
    complaint_buttons = []
    for complaint in complaint_types:
        traces = build_traces("All", complaint)
        complaint_buttons.append(dict(
            label=complaint[:40],           # truncate long names
            method="restyle",
            args=[{
                "x":          [t.x          for t in traces],
                "y":          [t.y          for t in traces],
                "customdata": [t.customdata for t in traces],
            }]
        ))

    fig.update_layout(
        title="Resolution Status by Borough — All Years | All Complaint Types",
        barmode="stack",
        xaxis_title="Borough",
        yaxis_title="Number of Complaints",
        legend_title="Status",
        height=600,
        updatemenus=[
            # Year dropdown
            dict(
                active=0,
                buttons=year_buttons,
                direction="down",
                showactive=True,
                x=0.5,
                xanchor="right",
                y=1.38,
                yanchor="top",
                pad={"r": 10}
            ),
            # Complaint Type dropdown
            dict(
                active=0,
                buttons=complaint_buttons,
                direction="down",
                showactive=True,
                x=1,
                xanchor="right",
                y=1.38,
                yanchor="top",
                pad={"r": 10}
            ),
        ],

    )

    return fig


def task8():
    df_temp = dataset.copy()
    df_temp["Created Date"] = pd.to_datetime(df_temp["Created Date"], errors="coerce")

    df_temp["Hour"]  = df_temp["Created Date"].dt.hour
    df_temp["Day"]   = df_temp["Created Date"].dt.day_name()
    df_temp["Month"] = df_temp["Created Date"].dt.month_name()
    df_temp["Year"]  = df_temp["Created Date"].dt.year.astype("Int64")

    day_order   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    month_order = ["January","February","March","April","May","June",
                   "July","August","September","October","November","December"]

    # ── 1. HOURLY: Hour vs Day of Week ──────────────────────────
    h1 = df_temp.groupby(["Day","Hour"]).size().reset_index(name="Count")
    h1["Day"] = pd.Categorical(h1["Day"], categories=day_order, ordered=True)
    h1 = h1.sort_values("Day")
    pivot_hourly = h1.pivot(index="Day", columns="Hour", values="Count").fillna(0)

    # ── 2. DAILY: Day of Week vs Month ──────────────────────────
    h2 = df_temp.groupby(["Day","Month"]).size().reset_index(name="Count")
    h2["Day"]   = pd.Categorical(h2["Day"],   categories=day_order,   ordered=True)
    h2["Month"] = pd.Categorical(h2["Month"], categories=month_order, ordered=True)
    h2 = h2.sort_values(["Day","Month"])
    pivot_daily = h2.pivot(index="Day", columns="Month", values="Count").fillna(0)

    # ── 3. MONTHLY: Month vs Year ────────────────────────────────
    h3 = df_temp.groupby(["Month","Year"]).size().reset_index(name="Count")
    h3["Month"] = pd.Categorical(h3["Month"], categories=month_order, ordered=True)
    h3 = h3.sort_values(["Month","Year"])
    pivot_monthly = h3.pivot(index="Month", columns="Year", values="Count").fillna(0)

    # ── 4. YEARLY: Year vs Borough ───────────────────────────────
    h4 = df_temp.groupby(["Year","Borough"]).size().reset_index(name="Count")
    h4 = h4.sort_values("Year")
    pivot_yearly = h4.pivot(index="Year", columns="Borough", values="Count").fillna(0)

    # ── Helper: build heatmap trace ──────────────────────────────
    def make_heatmap(pivot, xlabel, ylabel):
        return go.Heatmap(
            z=pivot.values.tolist(),
            x=[str(c) for c in pivot.columns],
            y=[str(r) for r in pivot.index],
            colorscale="YlOrRd",
            hovertemplate=(
                f"{xlabel}: " + "%{x}<br>" +
                f"{ylabel}: " + "%{y}<br>" +
                "Complaints: %{z:,}<extra></extra>"
            ),
            showscale=True,
            colorbar=dict(title="Complaints")
        )

    trace_hourly  = make_heatmap(pivot_hourly,  "Hour",    "Day")
    trace_daily   = make_heatmap(pivot_daily,   "Month",   "Day")
    trace_monthly = make_heatmap(pivot_monthly, "Year",    "Month")
    trace_yearly  = make_heatmap(pivot_yearly,  "Borough", "Year")

    fig = go.Figure(data=[trace_hourly, trace_daily, trace_monthly, trace_yearly])

    # hide all except first
    fig.data[0].visible = True
    fig.data[1].visible = False
    fig.data[2].visible = False
    fig.data[3].visible = False

    # ── Dropdown buttons ─────────────────────────────────────────
    patterns = [
        ("Hourly Pattern",  "Complaints by Hour of Day vs Day of Week",  "Hour of Day",  "Day of Week"),
        ("Daily Pattern",   "Complaints by Day of Week vs Month",         "Month",        "Day of Week"),
        ("Monthly Pattern", "Complaints by Month vs Year",                "Year",         "Month"),
        ("Yearly Pattern",  "Complaints by Year vs Borough",              "Borough",      "Year"),
    ]

    buttons = []
    for i, (label, title, xlabel, ylabel) in enumerate(patterns):
        visibility = [False] * 4
        visibility[i] = True
        buttons.append(dict(
            label=label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title":         title,
                    "xaxis.title":   xlabel,
                    "yaxis.title":   ylabel,
                }
            ]
        ))

    fig.update_layout(
        title="Complaints by Hour of Day vs Day of Week",
        xaxis_title="Hour of Day",
        yaxis_title="Day of Week",
        height=550,
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )],
        annotations=[
            dict(text="Pattern:", x=1.0, xref="paper", y=1.22,
                 yref="paper", showarrow=False, font=dict(size=12),
                 xanchor="right")
        ]
    )

    return fig

import plotly.graph_objects as go

def task_9():
    # Keep only required column and drop missing values
    df_temp = dataset[["Location Type"]].dropna()

    # Count occurrences
    location_counts = df_temp["Location Type"].value_counts()

    # Keep top 8
    top_8 = location_counts.nlargest(8)

    # Group into "Other"
    other_count = location_counts.iloc[8:].sum()

    # Combine
    final_counts = top_8.copy()
    if other_count > 0:
        final_counts["Other"] = other_count

    # Plot pie chart
    fig = go.Figure(data=[go.Pie(
        labels=final_counts.index,
        values=final_counts.values,
        hole=0,
        textinfo="label+percent",
        hoverinfo="label+value+percent"
    )])

    fig.update_layout(
        title="Distribution of Complaint Location Types (Top 8 + Other)",
        legend_title="Location Type",
        height=600
    )

    return fig


# ── layout: a grid of graph components ──
app.layout = html.Div([
    html.H1("NYC 311 Service Request Visual Analytics",
            style={"textAlign": "center", "color": "#1F4E79"}),

    # Row 1
    html.Div([
        dcc.Graph(figure=task1(), style={"width": "50%"}),
        dcc.Graph(figure=task2(), style={"width": "50%"}),
    ], style={"display": "flex"}),

    # Row 2
    html.Div([
        dcc.Graph(figure=task3(), style={"width": "50%"}),
        dcc.Graph(figure=task4(), style={"width": "50%"}),
    ], style={"display": "flex"}),

    # Row 3
    html.Div([
        dcc.Graph(figure=task5(), style={"width": "50%"}),
        dcc.Graph(figure=task6(), style={"width": "100%"}),  # map gets full width
    ], style={"display": "flex"}),

    # Row 4
    html.Div([
        dcc.Graph(figure=task7(), style={"width": "50%"}),
        dcc.Graph(figure=task8(), style={"width": "50%"}),
    ], style={"display": "flex"}),

    # Row 5
    html.Div([
        dcc.Graph(figure=task_9(), style={"width": "40%"}),
    ], style={"display": "flex", "justifyContent": "center"}),

], style={"fontFamily": "Calibri", "backgroundColor": "#F5F5F5", "padding": "20px"})

if __name__ == "__main__":
    app.run(debug=True)

/tmp/ipykernel_30318/3416243291.py:15: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv("https://huggingface.co/datasets/IshraNaznin/nyc311data2022to2024/resolve/main/data_2022_2024.csv")
/tmp/ipykernel_30318/3416243291.py:318: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = go.Figure(go.Choroplethmapbox(


<IPython.core.display.Javascript object>